In [1]:
import pandas as pd
import sys

sys.path.append('..')
from src.features.blocking import (
    make_block_keys,
    calculate_reduction_ratio,
    generate_candidate_pairs_from_block,
    evaluate_blocking_recall,
)

from src.features.block_cleaning import (
    build_stop_blocks,
    filter_stop_blocks,
    summarize_stop_blocks,
)

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_parquet("../data/processed/processed_dataset.parquet")
test_pairs = pd.read_parquet("../data/processed/test_pairs.parquet")

print(f"Dataset rows: {len(df):,}")
print(f"Test pairs: {len(test_pairs):,}")

Dataset rows: 3,655,635
Test pairs: 14,812


In [3]:
# ============================================================
# PREPARE TEST RECORDS
# ============================================================

test_indices = set(test_pairs["idx1"]) | set(test_pairs["idx2"])

df_test_records = df.loc[list(test_indices)].copy()

df_test_records = make_block_keys(
    df_test_records,
    name_col="name_latin",
)

print(f"Unique records in test pairs: {len(df_test_records):,}")

Unique records in test pairs: 21,811


In [4]:
# ============================================================
# BUILD STOP BLOCKS
# ============================================================

block_cols = [
    "block_prefix_len",
    "block_first_token_len",
    "block_sorted_tokens",
]

stop_block_tables = {}

for block_col in block_cols:
    print("=" * 60)
    print(block_col)
    print("=" * 60)

    stop_blocks = build_stop_blocks(
        df_test_records,
        block_col=block_col,
        max_block_size=500,
    )

    stop_block_tables[block_col] = stop_blocks

    summarize_stop_blocks(stop_blocks)

block_prefix_len
STOP BLOCK SUMMARY
Total blocks: 3,620

Stop block counts:
is_stop_block
False    3618
True        2
Name: count, dtype: int64

Large blocks:
2

Generic blocks:
0

Top stop blocks:


,block_prefix_len,block_size,is_large_block,is_generic_block,is_stop_block
0,ardy_6,822,True,False,True
1,prom_4,587,True,False,True


block_first_token_len
STOP BLOCK SUMMARY
Total blocks: 3,542

Stop block counts:
is_stop_block
False    3540
True        2
Name: count, dtype: int64

Large blocks:
2

Generic blocks:
0

Top stop blocks:


,block_first_token_len,block_size,is_large_block,is_generic_block,is_stop_block
0,ardy_6,822,True,False,True
1,prom_4,587,True,False,True


block_sorted_tokens
STOP BLOCK SUMMARY
Total blocks: 13,214

Stop block counts:
is_stop_block
False    13202
True        12
Name: count, dtype: int64

Large blocks:
2

Generic blocks:
11

Top stop blocks:


,block_sorted_tokens,block_size,is_large_block,is_generic_block,is_stop_block
0,ardyounaberakan by ynkeroutyoun,822,True,False,True
1,kompania promishlennaya,587,True,True,True
14,amp holding spy,28,False,True,True
18,limited ploutostar,21,False,True,True
34,amp holding,8,False,True,True
36,group migo,8,False,True,True
45,group of praym qompynis,7,False,True,True
170,energetik hrazdani hrazjek kazmakerpoutyoun,4,False,True,True
413,ar group invest,3,False,True,True
2424,energo holding invest,2,False,True,True


In [5]:
# ============================================================
# COMPARE BEFORE / AFTER STOP-BLOCK FILTERING
# ============================================================

cleaning_results = []

for block_col in block_cols:
    print("=" * 60)
    print(block_col)
    print("=" * 60)

    # before
    candidates_before = generate_candidate_pairs_from_block(
        df_test_records,
        block_col=block_col,
        max_block_size=500,
    )

    recall_before = evaluate_blocking_recall(
        labeled_pairs=test_pairs,
        candidate_pairs=candidates_before,
    )

    # after
    cleaned_records = filter_stop_blocks(
        df=df_test_records,
        block_col=block_col,
        stop_blocks=stop_block_tables[block_col],
    )

    candidates_after = generate_candidate_pairs_from_block(
        cleaned_records,
        block_col=block_col,
        max_block_size=500,
    )

    recall_after = evaluate_blocking_recall(
        labeled_pairs=test_pairs,
        candidate_pairs=candidates_after,
    )

    cleaning_results.append(
        {
            "block_col": block_col,
            "records_before": len(df_test_records),
            "records_after": len(cleaned_records),
            "candidates_before": len(candidates_before),
            "candidates_after": len(candidates_after),
            "positive_pairs": recall_before["positive_pairs"],
            "covered_before": recall_before["covered_positive_pairs"],
            "covered_after": recall_after["covered_positive_pairs"],
            "recall_before": recall_before["blocking_recall"],
            "recall_after": recall_after["blocking_recall"],
        }
    )

cleaning_results_df = pd.DataFrame(cleaning_results)

display(cleaning_results_df)

block_prefix_len
block_first_token_len
block_sorted_tokens


,block_col,records_before,records_after,candidates_before,candidates_after,positive_pairs,covered_before,covered_after,recall_before,recall_after
0,block_prefix_len,21811,20402,430237,430237,7305,7287,7287,0.997536,0.997536
1,block_first_token_len,21811,20402,427329,427329,7305,7287,7287,0.997536,0.997536
2,block_sorted_tokens,21811,20318,83528,82852,7305,7287,7287,0.997536,0.997536


In [6]:
# ============================================================
# MULTI-PASS BLOCKING WITH CLEANED BLOCKS
# ============================================================

candidate_frames = []

for block_col in block_cols:
    cleaned_records = filter_stop_blocks(
        df=df_test_records,
        block_col=block_col,
        stop_blocks=stop_block_tables[block_col],
    )

    candidates = generate_candidate_pairs_from_block(
        cleaned_records,
        block_col=block_col,
        max_block_size=500,
    )

    candidate_frames.append(candidates)

multi_candidates_clean = pd.concat(
    candidate_frames,
    ignore_index=True,
)

multi_candidates_clean["left"] = (
    multi_candidates_clean[["idx1", "idx2"]]
    .min(axis=1)
)

multi_candidates_clean["right"] = (
    multi_candidates_clean[["idx1", "idx2"]]
    .max(axis=1)
)

multi_candidates_clean["pair_key"] = (
    multi_candidates_clean["left"].astype(str)
    + "_"
    + multi_candidates_clean["right"].astype(str)
)

multi_candidates_clean = (
    multi_candidates_clean
    .drop_duplicates("pair_key")
    .reset_index(drop=True)
)

multi_recall_clean = evaluate_blocking_recall(
    labeled_pairs=test_pairs,
    candidate_pairs=multi_candidates_clean,
)

print("=" * 60)
print("CLEANED MULTI-PASS BLOCKING")
print("=" * 60)

print(f"Candidate pairs: {len(multi_candidates_clean):,}")
print(f"Positive pairs: {multi_recall_clean['positive_pairs']:,}")
print(f"Covered positives: {multi_recall_clean['covered_positive_pairs']:,}")
print(f"Blocking recall: {multi_recall_clean['blocking_recall']:.4%}")

CLEANED MULTI-PASS BLOCKING
Candidate pairs: 433,673
Positive pairs: 7,305
Covered positives: 7,287
Blocking recall: 99.7536%


In [7]:
# ============================================================
# SAVE RESULTS
# ============================================================

cleaning_results_df.to_parquet(
    "../data/processed/block_cleaning_results.parquet",
    index=False,
)

multi_candidates_clean.to_parquet(
    "../data/processed/test_blocking_candidates_clean.parquet",
    index=False,
)

for block_col, stop_blocks in stop_block_tables.items():
    stop_blocks.to_parquet(
        f"../data/processed/stop_blocks_{block_col}.parquet",
        index=False,
    )

print("Saved block cleaning artifacts.")

Saved block cleaning artifacts.


**Вывод:** Block cleaning уменьшает число кандидатов, но приводит к небольшой потере recall, поэтому в финальной версии предпочтение отдано multi-pass blocking без жесткого удаления stop-blocks.